# Collect pilot trajectories on Kaggle (T14) — local Llama 3.1 8B via Ollama

No Groq, no rate limits. The agent runs on Kaggle's GPU through Ollama.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings → Internet → **On**.

Run the cells in order. Outputs are saved under `/kaggle/working/`, which is kept when you click **Save Version**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # should show a Tesla T4

## 1. Install and start Ollama with Llama 3.1 8B

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"   # default context is too small for ReAct prompts
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"        # keep the model loaded between calls
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

!ollama pull llama3.1:8b
!ollama run llama3.1:8b "Say OK" --verbose 2>&1 | tail -5   # quick test + speed

## 2. Clone the repo and install (Python 3.10 venv — ALFWorld needs it)

In [ ]:
%cd /kaggle/working
!test -d tripwire-agent-monitor || git clone https://github.com/ManahilAyaz/tripwire-agent-monitor.git
%cd /kaggle/working/tripwire-agent-monitor
!git pull -q

!pip install -q uv
!test -d /kaggle/working/venv || uv venv --python 3.10 /kaggle/working/venv
!uv pip install -q --python /kaggle/working/venv/bin/python alfworld openai pyyaml
!uv pip install -q --python /kaggle/working/venv/bin/python -e .

## 3. Point the agent at Ollama instead of Groq

`make_groq_llm()` currently hard-codes Groq's URL. This cell makes it read `LLM_BASE_URL` / `LLM_API_KEY`
from the environment (falling back to Groq), and writes an Ollama agent config.

**Commit this change properly** (T14 follow-up commit, see the patch file) so the notebook doesn't need to patch it every time. This cell skips itself once the repo already has it.

In [ ]:
from pathlib import Path

p = Path("src/tripwire/agent/groq_client.py")
src = p.read_text()
old_key = 'api_key = os.environ.get("GROQ_API_KEY")'
old_client = 'client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=api_key)'
if "LLM_BASE_URL" not in src:
    src = src.replace(old_key, 'api_key = os.environ.get("LLM_API_KEY") or os.environ.get("GROQ_API_KEY")')
    src = src.replace(old_client,
        'base_url = os.environ.get("LLM_BASE_URL", "https://api.groq.com/openai/v1")\n'
        '    client = OpenAI(base_url=base_url, api_key=api_key)')
    p.write_text(src)
    print("Patched groq_client.py")
else:
    print("Repo already supports LLM_BASE_URL - no patch needed")

import yaml
cfg = yaml.safe_load(open("configs/agent.yaml"))
cfg["model"] = "llama3.1:8b"
yaml.safe_dump(cfg, open("configs/agent_ollama.yaml", "w"), sort_keys=False)
print(open("configs/agent_ollama.yaml").read())

os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"   # Ollama ignores it, the client just needs a value

## 4. ALFWorld game data + base config

In [ ]:
os.environ["ALFWORLD_DATA"] = "/kaggle/working/alfworld_data"
!test -d /kaggle/working/alfworld_data/json_2.1.1 || /kaggle/working/venv/bin/alfworld-download
!test -d /kaggle/working/alfworld || git clone -q https://github.com/alfworld/alfworld /kaggle/working/alfworld
!cp /kaggle/working/alfworld/configs/base_config.yaml .

## 5. (Only when resuming) restore episodes from a previous saved version

Kaggle keeps `/kaggle/working/` from the last **Save Version**. To continue in a new session:
**Add Input → Your Work → this notebook**, then run this cell. Skip it on the very first run.

In [ ]:
import glob, shutil
found = glob.glob("/kaggle/input/**/data/raw/pilot", recursive=True)
if found:
    shutil.copytree(found[0], "data/raw/pilot", dirs_exist_ok=True)
    print("Restored", len(os.listdir("data/raw/pilot")), "files from", found[0])
else:
    print("Nothing to restore - starting fresh")

## 6. Test run: 2 episodes (check it works and how long each takes)

In [ ]:
!time /kaggle/working/venv/bin/python scripts/run_batch.py --n 2 --config base_config.yaml --agent-config configs/agent_ollama.yaml --out-dir data/raw/pilot_test --seed 0
!ls data/raw/pilot_test

## 7. Full pilot: 40 episodes

Fresh folder: the old Qwen trajectories are **not** mixed in, so every pilot trajectory comes from the same model.
Resumable: if the session stops, rerun this cell (after step 5).

In [ ]:
!/kaggle/working/venv/bin/python scripts/run_batch.py --n 40 --config base_config.yaml --agent-config configs/agent_ollama.yaml --out-dir data/raw/pilot --seed 0

!zip -qr /kaggle/working/pilot_trajectories.zip data/raw/pilot
print("Done. Download pilot_trajectories.zip from the Output panel, or click Save Version to keep everything.")